In [1]:
import sys, json, warnings
sys.path.insert(0, "..")            # ml-service root, so `training` and `app` import
warnings.filterwarnings("ignore")
import duckdb, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from training import config as C
sns.set_theme(style="whitegrid")
IMG = C.ROOT_DIR / "docs" / "images"; IMG.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(); con.execute("SET memory_limit='1500MB'"); con.execute("SET threads=2")
G = str(C.FLIGHTS_GLOB).replace("\\", "/")
def q(sql): return con.execute(sql.replace("{G}", G)).df()
def save(name):
    plt.tight_layout(); plt.savefig(IMG / f"{name}.png", dpi=110); plt.show()


# 06 - Calibration and evaluation (production artifacts, test = Oct-Dec)

In [2]:
M = json.loads((C.METRICS_DIR / "model_metrics.json").read_text())
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
for ax_, (name, c) in zip(ax, M["calibration"].items()):
    for key, lab in (("curve_before", f"raw (Brier {c['brier_before']:.4f})"), ("curve_after", f"isotonic (Brier {c['brier_after']:.4f})")):
        pts = pd.DataFrame(c[key]); ax_.plot(pts.predicted, pts.observed, marker="o", label=lab)
    lim = max(pts.predicted.max(), 0.3); ax_.plot([0, lim], [0, lim], "k--", lw=1); ax_.set_title(f"{name} classifier calibration (test)")
    ax_.set_xlabel("predicted probability"); ax_.set_ylabel("observed frequency"); ax_.legend()
save("calibration_curves")

In [3]:
fig, ax = plt.subplots(figsize=(5.5, 5))
for name, pts in M["roc"].items():
    p = pd.DataFrame(pts); ax.plot(p.fpr, p.tpr, label=name)
ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate"); ax.set_title("ROC, delay task (test)"); ax.legend()
save("roc_curves")

In [4]:
xg = [m for m in M["models"] if m["model_name"] == "xgboost" and m["target"] == "delay"][0]
cm = np.array(xg["confusion_matrix"]); fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", xticklabels=["on time", "delayed"], yticklabels=["on time", "delayed"], ax=ax)
ax.set_xlabel("predicted"); ax.set_ylabel("actual"); ax.set_title(f"XGBoost delay, threshold {xg['threshold']:.2f}"); save("confusion_matrix_delay")

In [5]:
fi = pd.Series(M["feature_importance"]).head(15)[::-1]
fig, ax = plt.subplots(figsize=(7, 5)); fi.plot.barh(ax=ax, color="#3b82f6"); ax.set_title("XGBoost delay: top feature importances (gain share)"); save("feature_importance")

## Time split vs random split

In [6]:
r = M["random_split_experiment"]
pd.DataFrame({"honest time split (Oct-Dec)": xg["metrics"], "random 80/20 split": r["test"]["metrics"]}).loc[["roc_auc","pr_auc","f1","precision","recall"]].round(4)

,honest time split (Oct-Dec),random 80/20 split
roc_auc,0.6611,0.7451
pr_auc,0.2595,0.4655
f1,0.3296,0.4733
precision,0.2384,0.4098
recall,0.5336,0.5601


The random split is higher because every month is represented in training, so the model has seen the same seasonal delay levels and weather episodes as its test rows. The time split asks the real question: *can a model trained on Jan-Aug predict Oct-Dec?* Oct-Dec also has a much lower base rate (about 15% vs about 22%), which lowers precision and F1 for every model. We report the honest number.